# 03 — Customer Segmentation

*RFM-based k-means clustering: 7,903 customers → 4 statistically validated, business-actionable segments*

## Executive Summary

Takes the 7,903-customer RFM table from NB01 and groups them into **four behaviourally distinct segments** using k-means clustering. The optimal k is chosen through a four-metric voting system, cluster stability is confirmed over 50 bootstrap runs (mean ARI = 0.939), and segment separation is validated non-parametrically across all RFM dimensions.

Each segment exits with a revenue profile, churn risk breakdown, and a prioritised marketing action plan. Segment labels feed directly into NB04 (churn prediction) and NB05 (fraud risk cross-analysis).

| Output | Contents |
|---|---|
| `customer_segments.csv` | 7,903 customers with segment label and churn risk flag |
| `segment_profiles.json` | Full per-segment RFM metrics, revenue, and churn distribution |
| `segment_summary.csv` | Segment-level KPI roll-up |
| `marketing_recommendations.txt` | Per-segment prioritised action plans |

---

> **Three decisions worth noting downstream:**
> - RFM score columns were missing from the cached `rfm_df.parquet` and recalculated here via percentile rank (1–5) using the same method as NB01 — this upstream gap has been flagged
> - 26 pure-return customers (0.33% of cohort) were median-imputed before clustering; negligible impact on results
> - The 7.8× size imbalance across segments reflects real customer heterogeneity, not a model problem — k=4 is both metric-optimal and business-justified

In [1]:
# Standard imports
import pandas as pd
import numpy as np
import sys
import logging
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Project utilities
from n3a_utils import (
    setup_logger, load_config, get_project_root, set_run_id,
    print_section_header, get_output_paths, get_colors
)

# Data loading
from n3b_data_loader import load_data_for_segmentation

# RFM analysis
from n3c_rfm_analyzer import (
    analyze_rfm_distribution, create_rfm_segments, visualize_rfm_segments
)

# Feature preparation
from n3d_feature_prep import (
    prepare_clustering_features, validate_feature_distribution
)

# Cluster optimization
from n3e_cluster_optimizer import find_optimal_clusters

# Clustering
from n3f_clustering import (
    perform_kmeans_clustering, validate_clustering_stability,
    analyze_cluster_separation, get_cluster_centers
)

# Segment profiling
from n3g_segment_profiler import (
    create_segment_profiles, assign_segment_names, print_segment_summary
)

# Visualizations
from n3h_visualizations import (
    plot_segment_distribution, plot_rfm_by_segment,
    plot_pca_clusters, plot_segment_comparison
)

# Insights
from n3i_insights import (
    generate_segment_insights, create_marketing_recommendations
)

# Statistical validation
from n3j_statistical_tests import (
    validate_segment_quality, print_validation_report
)

# Export
from n3k_export import export_segment_data

# ── Redirect all log handlers to stdout ──────────────────────────────────
# This ensures logger output and print() share the same stream,
# so print_section_header() always appears before function log lines.
for _name in list(logging.Logger.manager.loggerDict.keys()) + ['root']:
    _lgr = logging.getLogger(_name) if _name != 'root' else logging.getLogger()
    for _h in _lgr.handlers:
        if isinstance(_h, logging.StreamHandler):
            _h.stream = sys.stdout

# Set up logging
logger = setup_logger(__name__)

# Generate run ID for this analysis
run_id = set_run_id()
logger.info(f"Starting customer segmentation analysis (run_id: {run_id})")

# Load configuration
config = load_config()
logger.info("Configuration loaded successfully")

RANDOM_STATE = config['notebook3']['random_state']
np.random.seed(RANDOM_STATE)
logger.info(f"Random seed set to {RANDOM_STATE}")

# Display settings
pd.set_option('display.max_columns', config['notebook3']['display']['max_columns'])
pd.set_option('display.precision', config['notebook3']['display']['precision'])
pd.set_option('display.float_format', config['notebook3']['display']['float_format'].format)

print(f"\nNotebook 03: Customer Segmentation")
print(f"Run ID: {run_id}")
print(f"Project Root: {get_project_root()}")

[8c30cec7] 2026-02-27 00:34:03 - __main__ - INFO - Starting customer segmentation analysis (run_id: 8c30cec7)
[8c30cec7] 2026-02-27 00:34:03 - __main__ - INFO - Configuration loaded successfully
[8c30cec7] 2026-02-27 00:34:03 - __main__ - INFO - Random seed set to 42



Notebook 03: Customer Segmentation
Run ID: 8c30cec7
Project Root: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project


## 1. Data Loading

Loads `enhanced_df.parquet` (transactions) and `rfm_df.parquet` (7,903 customers) from NB01. RFM score columns (`recency_score`, `frequency_score`, `monetary_score`) are absent from the cached parquet and recalculated here using the same inverted percentile-rank method as NB01. The 26 pure-return customers with non-finite RFM values are median-imputed before any further processing. Data integrity validation passes cleanly.

Note: I forgot to create an RFM score column in NB01, so I had to recalculate it here. The method is the same as NB01, so the scores are consistent, but this is an upstream gap that should be flagged for future runs.

In [2]:
print_section_header("DATA LOADING")

df, rfm_df = load_data_for_segmentation()

print("\nData Loaded:\n")
print(f"  Transactions: {len(df):,}")
print(f"  Customers:    {len(rfm_df):,}")
print(f"  Date range:   {df['order_date'].min()} to {df['order_date'].max()}")

print("\nRFM Data Preview:")
display(rfm_df.describe())

print("\nRFM Summary Statistics:")
display(rfm_df[['recency_days', 'frequency', 'monetary', 'loyalty_score']].describe())


                                  DATA LOADING                                  

[8c30cec7] 2026-02-27 00:34:03 - n3b_data_loader - INFO - Loading enhanced data from: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\data\processed\enhanced_df.parquet
[8c30cec7] 2026-02-27 00:34:03 - n3b_data_loader - INFO - Loaded 34,500 transactions
[8c30cec7] 2026-02-27 00:34:03 - n3b_data_loader - INFO - Date range: 2023-09-12 to 2025-09-11
[8c30cec7] 2026-02-27 00:34:03 - n3b_data_loader - INFO - Loaded RFM data for 7,903 customers
[8c30cec7] 2026-02-27 00:34:03 - n3b_data_loader - INFO - Missing RFM score columns: ['recency_score', 'frequency_score', 'monetary_score']
[8c30cec7] 2026-02-27 00:34:03 - n3b_data_loader - INFO - Auto-calculating scores using percentile rank method (1-5 scale)
[8c30cec7] 2026-02-27 00:34:03 - n3b_data_loader - INFO - Consider generating these scores in the RFM creation step (Notebook 01)
[8c30cec7] 2026-02-27 00:34:03 - n3b_data_loader -

,recency_days,frequency,monetary,net_monetary,avg_order_value,tenure_days,discount_usage_rate,category_diversity,return_rate,last_order_was_return,preferred_age,loyalty_score,churn,recency_score,frequency_score,monetary_score
count,7877.00,7877.00,7877.00,7903.00,7877.00,7903.00,7903.00,7877.00,7903.00,7903.00,7903.00,7877.00,7903.00,7903.00,7903.00,7903.00
mean,165.32,4.14,695.26,643.78,167.38,411.05,0.45,3.12,0.05,0.06,31.51,0.38,0.50,3.00,2.99,3.00
std,151.92,1.98,800.43,827.61,194.24,199.24,0.28,1.28,0.12,0.23,12.54,0.12,0.50,1.42,1.47,1.41
min,0.00,1.00,1.24,-5060.52,1.24,0.00,0.00,1.00,0.00,0.00,18.00,0.03,0.00,1.00,1.00,1.00
25%,49.00,3.00,189.44,159.36,57.28,277.00,0.25,2.00,0.00,0.00,22.00,0.32,0.00,2.00,2.00,2.00
50%,120.00,4.00,450.97,417.19,111.57,448.00,0.50,3.00,0.00,0.00,28.00,0.39,1.00,3.00,3.00,3.00
75%,236.00,5.00,903.80,869.33,206.61,573.00,0.67,4.00,0.00,0.00,38.00,0.46,1.00,4.00,4.00,4.00
max,728.00,12.00,13885.10,13885.10,3791.86,729.00,1.00,7.00,1.00,1.00,69.00,0.91,1.00,5.00,5.00,5.00



RFM Summary Statistics:


,recency_days,frequency,monetary,loyalty_score
count,7877.00,7877.00,7877.00,7877.00
mean,165.32,4.14,695.26,0.38
std,151.92,1.98,800.43,0.12
min,0.00,1.00,1.24,0.03
25%,49.00,3.00,189.44,0.32
50%,120.00,4.00,450.97,0.39
75%,236.00,5.00,903.80,0.46
max,728.00,12.00,13885.10,0.91


## 2. RFM Score Analysis

Score distributions are close to **uniform across all three dimensions** (each band 1–5 holds roughly 20% of customers) — the expected outcome of percentile-rank scoring. No floor or ceiling effects.

One exception: Frequency shows a mild bimodal pattern (21.9% at score 1, 23.4% at score 5). Expected — purchase counts are discrete and bounded, so customers naturally concentrate at the extremes with fewer in the middle.

> The rule-based RFM segments shown here (Loyal Customers 21.6%, At Risk 18.4%, Hibernating 16.4%, Potential Loyalists 15.4%) are a **baseline reference only** — fixed-threshold labels, not the segmentation output. The k-means results in Sections 4–8 are definitive.

In [3]:
print_section_header("RFM SCORE ANALYSIS")

import logging as _logging
_n3c = _logging.getLogger('n3c_rfm_analyzer')
_n3c_level = _n3c.level
_n3c.setLevel(_logging.WARNING)

analyze_rfm_distribution(rfm_df, config)
rfm_df = create_rfm_segments(rfm_df)
visualize_rfm_segments(rfm_df, config)

_n3c.setLevel(_n3c_level)
del _n3c, _n3c_level


                               RFM SCORE ANALYSIS                               


RFM Score Statistics:
       recency_score  frequency_score  monetary_score
count        7903.00          7903.00         7903.00
mean            3.00             2.99            3.00
std             1.42             1.47            1.41
min             1.00             1.00            1.00
25%             2.00             2.00            2.00
50%             3.00             3.00            3.00
75%             4.00             4.00            4.00
max             5.00             5.00            5.00

Score Distribution (1=Worst, 5=Best):

Recency Score:
  1:  1,583 ( 20.0%)
  2:  1,583 ( 20.0%)
  3:  1,583 ( 20.0%)
  4:  1,550 ( 19.6%)
  5:  1,604 ( 20.3%)

Frequency Score:
  1:  1,729 ( 21.9%)
  2:  1,528 ( 19.3%)
  3:  1,566 ( 19.8%)
  4:  1,227 ( 15.5%)
  5:  1,853 ( 23.4%)

Monetary Score:
  1:  1,580 ( 20.0%)
  2:  1,581 ( 20.0%)
  3:  1,580 ( 20.0%)
  4:  1,581 ( 20.0%)
  5:  1,581 ( 20.0%)

RF

## 3. Feature Preparation

Five features selected for clustering to capture distinct dimensions of customer behaviour:

| Feature | Dimension |
|---|---|
| `recency_days` | Engagement timing — how recently active |
| `frequency` | Purchase volume — how often they buy |
| `monetary` | Total spend — overall revenue contribution |
| `avg_order_value` | Spend efficiency — revenue per transaction |
| `loyalty_score` | Composite engagement — recency + frequency + monetary weighted |

All five features standardised with StandardScaler (mean ≈ 0, std ≈ 1). Post-scaling checks confirm no missing or infinite values.

**On correlated features:** Collinearity is acceptable in clustering because the objective is distance-based separation rather than coefficient interpretation. Clustering performed using Euclidean distance.

- `loyalty_score` is intentionally correlated with `frequency` (r = 0.865) and `recency_days` (r = −0.812) since it is derived from both.
- `monetary` and `avg_order_value` are strongly correlated (r = 0.784), but they distinguish *volume-driven* from *value-driven* customers, which is important for meaningful segment interpretation. As a result, both features are retained.

> All missing values for the 26 return-only customers across 5 features (130 total) were successfully filled with median values and contain no invalid numbers.

In [4]:
print_section_header("FEATURE PREPARATION")

# Get feature list from config
feature_list = config.get('notebook3', {}).get('rfm_analyzer', {}).get(
    'default_features',
    ['recency_days', 'frequency', 'monetary', 'avg_order_value', 'loyalty_score']
)

X_scaled, feature_names, clust_df = prepare_clustering_features(
    rfm_df,
    feature_list=feature_list
)

# Validate feature distributions
validation = validate_feature_distribution(X_scaled, feature_names)

print(f"\nFeatures: {feature_list}")
print(f"Scaled feature matrix: {X_scaled.shape}")
print(f"Clustering DataFrame:   {clust_df.shape}")


                              FEATURE PREPARATION                               

[8c30cec7] 2026-02-27 00:34:04 - n3d_feature_prep - INFO - Using 5 features: ['recency_days', 'frequency', 'monetary', 'avg_order_value', 'loyalty_score']
[8c30cec7] 2026-02-27 00:34:04 - n3d_feature_prep - WARNING - Found 130 missing values across all features
[8c30cec7] 2026-02-27 00:34:04 - n3d_feature_prep - INFO - Imputed 26 NaNs in 'recency_days' with median 120.00
[8c30cec7] 2026-02-27 00:34:04 - n3d_feature_prep - INFO - Imputed 26 NaNs in 'frequency' with median 4.00
[8c30cec7] 2026-02-27 00:34:04 - n3d_feature_prep - INFO - Imputed 26 NaNs in 'monetary' with median 450.97
[8c30cec7] 2026-02-27 00:34:04 - n3d_feature_prep - INFO - Imputed 26 NaNs in 'avg_order_value' with median 111.57
[8c30cec7] 2026-02-27 00:34:04 - n3d_feature_prep - INFO - Imputed 26 NaNs in 'loyalty_score' with median 0.39
[8c30cec7] 2026-02-27 00:34:04 - n3d_feature_prep - INFO - Data quality checks passed: no NaNs or Infs

## 4. Optimal Cluster Determination

k evaluated over [2–8] using four independent methods, each voting for their preferred cluster count:

| Method | Preferred k | Score |
|---|---|---|
| Silhouette ↑ | k = 3 | 0.3530 |
| Davies-Bouldin ↓ | k = 4 | 0.9884 |
| Calinski-Harabasz ↑ | k = 4 | 4,188.61 |
| Elbow | k = 4 | Sharpest inertia drop |

**k = 4 selected — 3 of 4 methods agree.**

Silhouette's preference for k = 3 is acknowledged but deliberately overridden. At k = 3, the High-Value at Risk segment (6.1% of customers, 24.7% of revenue) is absorbed into a larger group — erasing the most commercially significant signal in the dataset. Davies-Bouldin, Calinski-Harabasz, and the elbow method all point to k = 4, and the business case strongly supports the more granular split.

In [5]:
print_section_header("OPTIMAL CLUSTER DETERMINATION")

optimal_k, metrics_df = find_optimal_clusters(
    X_scaled,
    config=config
)

print(f"\nOptimal K: {optimal_k}")
print("\nMetrics Summary:")
display(metrics_df)


                         OPTIMAL CLUSTER DETERMINATION                          

[8c30cec7] 2026-02-27 00:34:04 - n3e_cluster_optimizer - INFO - Testing k values: [2, 3, 4, 5, 6, 7, 8]
[8c30cec7] 2026-02-27 00:34:04 - n3e_cluster_optimizer - INFO - Samples: 7,903, Features: 5
[8c30cec7] 2026-02-27 00:34:04 - n3e_cluster_optimizer - INFO - Evaluating k=2
[8c30cec7] 2026-02-27 00:34:06 - n3e_cluster_optimizer - INFO - Evaluating k=3
[8c30cec7] 2026-02-27 00:34:07 - n3e_cluster_optimizer - INFO - Evaluating k=4
[8c30cec7] 2026-02-27 00:34:08 - n3e_cluster_optimizer - INFO - Evaluating k=5
[8c30cec7] 2026-02-27 00:34:09 - n3e_cluster_optimizer - INFO - Evaluating k=6
[8c30cec7] 2026-02-27 00:34:10 - n3e_cluster_optimizer - INFO - Evaluating k=7
[8c30cec7] 2026-02-27 00:34:11 - n3e_cluster_optimizer - INFO - Evaluating k=8
[8c30cec7] 2026-02-27 00:34:12 - n3e_cluster_optimizer - INFO - Determining optimal k using voting system
[8c30cec7] 2026-02-27 00:34:12 - n3e_cluster_optimizer - INFO 

,k,inertia,silhouette,davies_bouldin,calinski_harabasz
0,2,26487.58,0.32,1.18,3885.98
1,3,19880.62,0.35,1.02,3901.09
2,4,15251.98,0.33,0.99,4188.61
3,5,13542.92,0.32,1.05,3786.65
4,6,11929.86,0.29,1.04,3652.02
5,7,10714.46,0.29,1.00,3537.42
6,8,9685.01,0.27,1.02,3473.83


## 5. K-Means Clustering

K-means fit on the 7,903 × 5 scaled feature matrix (k = 4, `random_state = 42`, 50 initialisations). Converged in **15 iterations** — well within the 300-iteration ceiling, indicating stable, well-defined clusters.

| Cluster | Segment | n | Share |
|---|---|---|---|
| 0 | Needs Engagement | 3,766 | 47.7% |
| 1 | Loyal Customers | 2,215 | 28.0% |
| 2 | Lost Customers | 1,442 | 18.2% |
| 3 | High-Value at Risk | 480 | 6.1% |

The size imbalance is by design — real customer bases are not evenly distributed across behaviour types, and forcing balance would destroy meaningful segment boundaries. The 7.8× ratio between the largest and smallest segments reflects genuine heterogeneity in the customer base.

In [6]:
print_section_header("K-MEANS CLUSTERING")

# Get kmeans config
kmeans_cfg = config.get('notebook3', {}).get('clustering', {}).get('kmeans', {})

kmeans, cluster_labels = perform_kmeans_clustering(
    X_scaled,
    n_clusters=optimal_k,
    random_state=kmeans_cfg.get('random_state', 42),
    n_init=kmeans_cfg.get('n_init', 50),
    max_iter=kmeans_cfg.get('max_iter', 300)
)

# Add cluster labels to clust_df (clean features)
clust_df['cluster'] = cluster_labels

# Also add to rfm_df for backward compatibility
# Merge via customer_id to ensure alignment
rfm_df = rfm_df.drop(columns=['cluster'], errors='ignore')
rfm_df = rfm_df.merge(
    clust_df[['customer_id', 'cluster']],
    on='customer_id',
    how='left'
)

print(f"\nClustering complete: {optimal_k} segments created")


                               K-MEANS CLUSTERING                               

[8c30cec7] 2026-02-27 00:34:13 - n3f_clustering - INFO - Clustering complete (inertia=15251.98, iterations=15)
[8c30cec7] 2026-02-27 00:34:13 - n3f_clustering - INFO - Cluster size distribution:
[8c30cec7] 2026-02-27 00:34:13 - n3f_clustering - INFO -   Cluster 0:  3,766 ( 47.7%)
[8c30cec7] 2026-02-27 00:34:13 - n3f_clustering - INFO -   Cluster 1:  2,215 ( 28.0%)
[8c30cec7] 2026-02-27 00:34:13 - n3f_clustering - INFO -   Cluster 2:  1,442 ( 18.2%)
[8c30cec7] 2026-02-27 00:34:13 - n3f_clustering - INFO -   Cluster 3:    480 (  6.1%)
[8c30cec7] 2026-02-27 00:34:13 - n3f_clustering - WARNING - Clusters imbalanced (ratio=7.8)

Clustering complete: 4 segments created


## 6. Clustering Quality Validation

**Stability — Excellent.**
K-means re-run 50 times with different random seeds. Mean ARI = 0.9388 (std = 0.039, min = 0.8250, max = 0.9845). An ARI near 1.0 means cluster assignments are nearly identical regardless of initialisation — the solution is robust, not a product of a lucky seed.

**Separation — Good overall, with one expected exception.**
Overall Silhouette score: 0.3285.

| Segment | Silhouette | Rating |
|---|---|---|
| Needs Engagement | 0.356 | Good |
| Lost Customers | 0.345 | Good |
| Loyal Customers | 0.312 | Good |
| High-Value at Risk | 0.141 | Fair |

The lower score for High-Value at Risk is **expected and informative, not a flaw**. High-value customers share high spend but vary widely in recency and frequency — their internal heterogeneity is precisely why they need their own segment rather than being absorbed into Loyal Customers.

In [7]:
print_section_header("CLUSTERING QUALITY VALIDATION")

# Get stability config
stability_cfg = config.get('notebook3', {}).get('clustering', {}).get('stability', {})

# Stability validation
mean_ari = validate_clustering_stability(
    X_scaled,
    n_clusters=optimal_k,
    n_iterations=stability_cfg.get('n_iterations', 50),
    random_state=kmeans_cfg.get('random_state', 42),
    sample_fraction=stability_cfg.get('sample_fraction', 0.8)
)

# Separation analysis
separation_metrics = analyze_cluster_separation(X_scaled, cluster_labels)

# Cluster centers
centers_df = get_cluster_centers(kmeans, feature_names)
print("\nCluster Centers:")
display(centers_df)


                         CLUSTERING QUALITY VALIDATION                          

[8c30cec7] 2026-02-27 00:34:13 - n3f_clustering - INFO - Completed 10/50 iterations
[8c30cec7] 2026-02-27 00:34:14 - n3f_clustering - INFO - Completed 20/50 iterations
[8c30cec7] 2026-02-27 00:34:14 - n3f_clustering - INFO - Completed 30/50 iterations
[8c30cec7] 2026-02-27 00:34:14 - n3f_clustering - INFO - Completed 40/50 iterations
[8c30cec7] 2026-02-27 00:34:15 - n3f_clustering - INFO - Completed 50/50 iterations
[8c30cec7] 2026-02-27 00:34:15 - n3f_clustering - INFO - Stability Results:
[8c30cec7] 2026-02-27 00:34:15 - n3f_clustering - INFO - Mean ARI: 0.9392
[8c30cec7] 2026-02-27 00:34:15 - n3f_clustering - INFO - Std Dev: 0.0390
[8c30cec7] 2026-02-27 00:34:15 - n3f_clustering - INFO - Min/Max: 0.8250 / 0.9845
[8c30cec7] 2026-02-27 00:34:15 - n3f_clustering - INFO - Stability Assessment: Excellent
[8c30cec7] 2026-02-27 00:34:16 - n3f_clustering - INFO - Cluster Separation Analysis:
[8c30cec7] 2026-0

,recency_days,frequency,monetary,avg_order_value,loyalty_score
cluster,,,,,
0,-0.27,-0.33,-0.38,-0.26,-0.10
1,-0.59,1.16,0.40,-0.02,1.03
2,1.65,-0.97,-0.52,-0.21,-1.51
3,-0.08,0.16,2.65,2.78,0.53


## 7. Statistical Validation

Levene's test confirmed unequal variances across all features (p < 0.001), ruling out ANOVA. Kruskal-Wallis non-parametric tests used throughout — consistent with the distributional findings in NB01.

| Feature | H-statistic | η² (Effect Size) | Verdict |
|---|---|---|---|
| `loyalty_score` | 6,009.40 | 0.739 | Large |
| `recency_days` | 3,673.71 | 0.628 | Large |
| `monetary` | 3,679.67 | 0.589 | Large |
| `frequency` | 4,875.82 | 0.602 | Large |

All four features significant (p < 0.001) with large effect sizes. Dunn's post-hoc tests (Bonferroni-corrected) confirm 6/6 significant pairwise comparisons for `frequency`, `monetary`, and `loyalty_score`; 5/6 for `recency_days`. The one non-significant recency pair reflects genuine behavioural overlap between two adjacent segments — expected, and does not undermine the segmentation.

**All four segments are statistically valid and meaningfully distinct.**

In [8]:
print_section_header("STATISTICAL VALIDATION")

# clust_df has imputed, clean features and cluster assignments
validation_results = validate_segment_quality(
    segment_df=clust_df,
    continuous_features=['recency_days', 'frequency', 'monetary', 'loyalty_score'],
    categorical_features=None,
    alpha=0.05
)

print_validation_report(validation_results)

print(f"\nValidation Result: {validation_results['verdict']}")


                             STATISTICAL VALIDATION                             

[8c30cec7] 2026-02-27 00:34:16 - n3j_statistical_tests - INFO - Levene's Test for recency_days:
[8c30cec7] 2026-02-27 00:34:16 - n3j_statistical_tests - INFO - W-statistic: 257.91
[8c30cec7] 2026-02-27 00:34:16 - n3j_statistical_tests - INFO - P-value: 0.0000
[8c30cec7] 2026-02-27 00:34:16 - n3j_statistical_tests - INFO - [WARN] Variances are NOT homogeneous - consider non-parametric tests
[8c30cec7] 2026-02-27 00:34:16 - n3j_statistical_tests - INFO - Testing feature: recency_days
[8c30cec7] 2026-02-27 00:34:16 - n3j_statistical_tests - INFO - Kruskal-Wallis Test for recency_days:
[8c30cec7] 2026-02-27 00:34:16 - n3j_statistical_tests - INFO - H-statistic: 3673.71
[8c30cec7] 2026-02-27 00:34:16 - n3j_statistical_tests - INFO - P-value: 0.0000
[8c30cec7] 2026-02-27 00:34:16 - n3j_statistical_tests - INFO - [PASS] Segments have significantly different distributions (p < 0.05)
[8c30cec7] 2026-02-27 00:34:1

## 8. Segment Profiling

Segments ranked by revenue contribution and named by their dominant behavioural signature:

---

### 🟢 Loyal Customers — *Protect & Deepen*
**28.0% of customers · 41.1% of revenue ($2,248,394)**

The highest-LTV segment. Mean recency 76 days (median 56), frequency 6.4 orders, monetary $1,015 avg. 52.9% are fully active — the healthiest churn profile in the portfolio.

However, **47.1% carry some level of risk** (At-Risk 25.6%, Inactive 12.7%, Churned 8.8%). Even this segment requires active retention investment. Loyalty is not self-sustaining.

---

### 🟡 Needs Engagement — *Activate or Lose*
**47.7% of customers · 26.9% of revenue ($1,475,163)**

The largest segment, but churn status is evenly split — Active 28.3%, Churned 25.9%, At-Risk 23.7%, Inactive 22.1%. Mean recency 124 days, frequency 3.5 orders, monetary $394 avg. No single dominant status means these customers are **drifting**, not decisively churned. Without intervention, the natural trajectory is into Lost Customers.

---

### 🔴 High-Value at Risk — *Urgent Recovery*
**6.1% of customers · 24.7% of revenue ($1,351,466)**

The highest-priority segment despite its small size. Monetary $2,816 avg (median $2,542), AOV $707 — **4.2× the portfolio average**. Mean recency 153 days (median 116), frequency 4.5 orders. 32.7% churned. Fewer than 500 customers control nearly a quarter of total revenue — loss of even a fraction has a disproportionate P&L impact.

---

### ⚫ Lost Customers — *Triage Only*
**18.2% of customers · 7.3% of revenue ($401,514)**

Mean recency 415 days (median 392), frequency 2.2 orders, 100% churned by definition. Re-engagement ROI is low. The more valuable strategic question is upstream: understanding what drives migration from Needs Engagement into this segment is worth more than mass win-back attempts.

In [9]:
print_section_header("SEGMENT PROFILING")

# Create segment profiles
segment_profiles = create_segment_profiles(clust_df, rfm_df)

# Assign meaningful names
segment_names = assign_segment_names(segment_profiles)

# Print summary
print_segment_summary(segment_profiles, segment_names)


                               SEGMENT PROFILING                                

[8c30cec7] 2026-02-27 00:34:16 - n3g_segment_profiler - INFO - Cluster 0: 3,766 customers (47.7%), $1,475,163 revenue, Active dominant
[8c30cec7] 2026-02-27 00:34:16 - n3g_segment_profiler - INFO - Cluster 1: 2,215 customers (28.0%), $2,248,394 revenue, Active dominant
[8c30cec7] 2026-02-27 00:34:16 - n3g_segment_profiler - INFO - Cluster 2: 1,442 customers (18.2%), $401,514 revenue, Churned dominant
[8c30cec7] 2026-02-27 00:34:16 - n3g_segment_profiler - INFO - Cluster 3: 480 customers (6.1%), $1,351,466 revenue, Churned dominant
[8c30cec7] 2026-02-27 00:34:16 - n3g_segment_profiler - INFO - Cluster 3: High-Value at Risk (R=3.0, F=3.2, M=5.0, Churned, Revenue=24.7%)
[8c30cec7] 2026-02-27 00:34:16 - n3g_segment_profiler - INFO - Cluster 1: Loyal Customers (R=3.9, F=4.7, M=4.1, Active, Revenue=41.1%)
[8c30cec7] 2026-02-27 00:34:16 - n3g_segment_profiler - INFO - Cluster 0: Needs Engagement (R=3.2, F=2.5, 

## 9. Segment Visualisations

Four interactive Plotly charts saved to `outputs/figures/notebook3_fig/`:

| File | What it shows |
|---|---|
| `segment_distribution.html` | Customer count and revenue share by segment — the Pareto structure at a glance |
| `rfm_by_segment.html` | RFM metric distributions per segment (boxplots) — median and spread per dimension |
| `pca_clusters.html` | 2D PCA projection — visual confirmation of cluster separation in reduced feature space |
| `segment_comparison.html` | Cross-segment KPI comparison dashboard — side-by-side profile for all four segments |

All charts are self-contained HTML — open in any browser without a Python environment.

In [10]:
print_section_header("SEGMENT VISUALIZATIONS")

plot_segment_distribution(rfm_df, segment_names, config)
plot_rfm_by_segment(rfm_df, segment_names, config)
plot_pca_clusters(X_scaled, cluster_labels, segment_names, config)
plot_segment_comparison(rfm_df, df, segment_names, config)

print("\nAll visualizations saved to outputs/figures/notebook3_fig/")


                             SEGMENT VISUALIZATIONS                             

[8c30cec7] 2026-02-27 00:34:16 - n3h_visualizations - INFO - Saved segment distribution: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\figures\notebook3_fig\segment_distribution.html
[8c30cec7] 2026-02-27 00:34:16 - n3h_visualizations - INFO - Saved RFM metrics: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\figures\notebook3_fig\rfm_by_segment.html
[8c30cec7] 2026-02-27 00:34:17 - n3h_visualizations - INFO - Saved PCA visualization: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\figures\notebook3_fig\pca_clusters.html
[8c30cec7] 2026-02-27 00:34:17 - n3h_visualizations - INFO - Saved segment comparison: c:\Users\Rich Justine Gambe\Documents\AAA Projects\ecommerce-analytics-project\outputs\figures\notebook3_fig\segment_comparison.html

All visualizations saved to outputs/figures/notebo

## 10. Business Insights & Marketing Playbook

Action priorities ordered by revenue at stake and urgency:

---

**🔴 High-Value at Risk — Urgent Recovery**

Deploy immediate, high-touch outreach. Time-bounded exclusive offers (20–25% discount + premium gift) are justified — re-engaging a customer worth $2,816 in average spend is commercially sound even at significant per-customer cost. Target the 32.7% churned sub-cohort first. This segment also carries the highest fraud risk in NB05 (7.5% High/Critical) — recovery and risk interventions should be coordinated, not siloed.

---

**🟢 Loyal Customers — Protect & Deepen**

Build a tiered loyalty programme to grow share-of-wallet. Target AOV uplift from $163 toward $200+ through product bundles and curated recommendations. Use milestone and anniversary rewards to reinforce loyalty identity. The **34.4% at-risk or churned sub-cohort** is the immediate retention priority — these are high-LTV customers beginning to drift and are easier to retain now than to recover later.

---

**🟡 Needs Engagement — Activate Before They Drift**

Run upsell and cross-sell campaigns targeting the 28.3% active sub-cohort. Use loyalty points and personalised recommendations to push frequency from 3.5 toward the Loyal segment's 6.4. The even churn distribution signals early-stage drift — structured re-engagement now is substantially cheaper than recovery once they reach Lost Customers.

---

**⚫ Lost Customers — Low-Cost Triage Only**

Automated win-back emails only — no paid spend. The strategic priority is upstream: understanding *why* customers migrate from Needs Engagement into this segment generates more long-term value than mass recovery attempts. 94.3% are confirmed churned by NB04's forward-looking 180-day definition.

In [11]:
print_section_header("BUSINESS INSIGHTS")

segment_insights = generate_segment_insights(
    profiles=segment_profiles,
    segment_names=segment_names,
    max_insights=3
)

marketing_recommendations = create_marketing_recommendations(
    profiles=segment_profiles,
    segment_names=segment_names,
    max_recommendations=5
)

print("\nSegment Insights and Recommendations:")
print("=" * 80)

for cluster_id in sorted(segment_profiles.keys()):
    name = segment_names[cluster_id]
    print(f"\n{name} (Cluster {cluster_id})")
    print("-" * 80)

    print("\nKey Characteristics:")
    for char in segment_insights[cluster_id]['characteristics']:
        print(f"  - {char}")

    print("\nStrategic Insights:")
    for insight in segment_insights[cluster_id]['insights']:
        print(f"  - {insight}")

    print("\nMarketing Recommendations:")
    for i, rec in enumerate(marketing_recommendations[cluster_id], 1):
        print(f"  {i}. {rec}")


                               BUSINESS INSIGHTS                                

[8c30cec7] 2026-02-27 00:34:17 - n3i_insights - INFO - Generated insights for 4 segments (max 3 per segment)
[8c30cec7] 2026-02-27 00:34:17 - n3i_insights - INFO - Created recommendations for 4 segments (max 5 per segment)

Segment Insights and Recommendations:

Needs Engagement (Cluster 0)
--------------------------------------------------------------------------------

Key Characteristics:
  - Size: 47.7% (3,766 customers)
  - Revenue: 26.9% ($1,475,163)
  - Churn Risk: Active (50% at-risk/churned)

Strategic Insights:

Marketing Recommendations:
  1. Upsell and cross-sell campaigns
  2. Loyalty points or rewards program

Loyal Customers (Cluster 1)
--------------------------------------------------------------------------------

Key Characteristics:
  - Size: 28.0% (2,215 customers)
  - Revenue: 41.1% ($2,248,394)
  - Churn Risk: Active (34% at-risk/churned)

Strategic Insights:

Marketing Recommendat

## 11. Export Results

Four files written to `data/processed/`:

| File | Contents |
|---|---|
| `customer_segments.csv` | 7,903 records with segment label and churn risk flag |
| `segment_profiles.json` | Full per-segment metrics for downstream use or reporting |
| `segment_summary.csv` | KPI roll-up: size, revenue, RFM medians per segment |
| `marketing_recommendations.txt` | Per-segment prioritised action plans |

---

**Downstream consumption:**

| Notebook | Consumes | Purpose |
|---|---|---|
| NB04 — Churn Prediction | `customer_segments.csv` | Segment × churn rate cross-validation |
| NB05 — Fraud Detection | `customer_segments.csv` | Segment × fraud tier overlap analysis |

`customer_segments.csv` is the primary handoff artifact. Segment membership is the validated churn-risk proxy for production use (see NB04, Section 5) and the basis for compounded-risk identification in NB05.

In [12]:
print_section_header("EXPORT RESULTS")

export_paths = export_segment_data(
    rfm_df=rfm_df,
    segment_profiles=segment_profiles,
    segment_names=segment_names,
    recommendations=marketing_recommendations
)

print("\nExported Files:")
for file_type, file_path in export_paths.items():
    print(f"  - {file_type}: {file_path}")

print("\nSegmentation analysis complete!")


                                 EXPORT RESULTS                                 

[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Exporting customer segment assignments
[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Added churn_risk column based on recency thresholds
[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Added legacy churn column from RFM data
[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Saved: customer_segments.csv (7,903 records)
[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Exporting segment profiles
[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Saved: segment_profiles.json (4 segments)
[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Exporting marketing recommendations
[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Saved: marketing_recommendations.txt
[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Exporting segment summary
[8c30cec7] 2026-02-27 00:34:17 - n3k_export - INFO - Saved: segment_summary.csv
[8c30cec7] 2026-